# Amazon Bedrock Reinforcement Fine-Tuning — Nemotron Nano 3 30B on Codeforces

This notebook demonstrates reinforcement fine-tuning (RFT) of **NVIDIA Nemotron Nano 3 30B** on competitive programming problems from the [Codeforces-Python-Submissions](https://huggingface.co/datasets/MatrixStudio/Codeforces-Python-Submissions) dataset, using Bedrock's OpenAI-compatible fine-tuning APIs.

## What's RFT?

Traditional fine-tuning shows a model examples and says "produce outputs like this." RFT takes a different approach: it lets the model generate its own responses, then uses a reward signal to reinforce good outputs and discourage bad ones. Think of it like training with a coach who gives feedback rather than just copying from a textbook.

For competitive programming, this works well because we can automatically verify code correctness by running it against test cases — no human labeling needed.

## What's the Codeforces Dataset?

The [MatrixStudio/Codeforces-Python-Submissions](https://huggingface.co/datasets/MatrixStudio/Codeforces-Python-Submissions) dataset contains ~621K training samples of Codeforces competitive programming problems with Python solutions. Each sample includes:
- Problem description with input/output specifications
- Example test cases and hidden test cases
- Accepted Python solutions
- Problem difficulty ratings and tags

We filter for accepted solutions on problems rated 800–1600 (beginner to intermediate) to create a focused training set.

## API Operations Covered

1. Download and preprocess the Codeforces dataset from HuggingFace
2. Format data as OpenAI-compatible JSONL for RFT
3. Deploy a Lambda reward function that runs code against test cases
4. Upload training data and create an RFT job
5. Monitor training and run inference

**Prerequisites:**
- AWS credentials with permissions for IAM, Lambda, and Bedrock
- Bedrock model access for Nemotron Nano 3 30B (`nvidia.nemotron-nano-3-30b`)
- Python 3.11+

---
## Step 1: Install Required Dependencies

In [ ]:
%%capture install_output

!pip install --upgrade boto3 botocore
!pip install --upgrade openai
!pip install --upgrade httpx
!pip install --upgrade colorama tiktoken aws-bedrock-token-generator
!pip install --upgrade datasets huggingface_hub

print("Dependencies installed successfully")

In [ ]:
import boto3
import openai
import httpx
import datasets

print(f"boto3 version: {boto3.__version__}")
print(f"openai version: {openai.__version__}")
print(f"httpx version: {httpx.__version__}")
print(f"datasets version: {datasets.__version__}")
print("\nAll imports successful")

---
## Step 2: Configuration

### Bedrock API Keys

Before we can proceed, please use the following documentation to generate a short- or long-term Bedrock API Key:

- https://docs.aws.amazon.com/bedrock/latest/userguide/api-keys.html
- https://docs.aws.amazon.com/bedrock/latest/userguide/api-keys-generate.html#api-keys-generate-console

Here we use the Bedrock token generator library to create a short-term key.

### Fine-tuning role

Create a role (or edit your SageMaker execution role) with the following permissions:

- **Lambda invocation** — allow `lambda:InvokeFunction` on your reward function ARN
- **Bedrock model access** — allow `bedrock:InvokeModel` on foundation models (for AI-as-judge if needed)

In [ ]:
import sys
import os
import json
import time

# Add the project root (two levels up) to the Python path so we can import helpers
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

from helpers import (
    create_lambda_deployment_package,
    cleanup_lambda_deployment_package
)

# ============== UPDATE THESE VALUES ==============
AWS_REGION = "us-west-2"
AWS_PROFILE = None  # Set to your profile name, or None for default credentials
# =================================================

# Create session
session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION) if AWS_PROFILE else boto3.Session(region_name=AWS_REGION)
AWS_ACCOUNT_ID = session.client('sts').get_caller_identity()['Account']

# Dataset configuration
DATASET_NAME = "codeforces"
HF_DATASET = "MatrixStudio/Codeforces-Python-Submissions"

# Resource names
LAMBDA_FUNCTION_NAME = f"{DATASET_NAME}-reward-function"
LAMBDA_ROLE_NAME = f"{DATASET_NAME.upper()}-Lambda-Role"
BEDROCK_ROLE_NAME = "BedrockRFTRole"
REWARD_FUNCTION_FILE = "../../reward-functions/codeforces_rew_func.py"
REWARD_FUNCTION_MODULE = "codeforces_rew_func"

print(f"Account: {AWS_ACCOUNT_ID}")
print(f"Region:  {AWS_REGION}")
print(f"Dataset: {HF_DATASET}")

---
## Step 3: Download & Preprocess the Codeforces Dataset

We download the dataset from HuggingFace, then filter and format it for RFT:

1. **Filter** for accepted (`verdict == "OK"`) solutions on problems rated 800–1600
2. **Deduplicate** by problem (keep one solution per unique problem)
3. **Build prompts** from the problem description, input/output specs, and examples
4. **Format** as OpenAI-compatible JSONL with `messages` and `metadata` (containing test cases for the reward function)
5. **Sample** a manageable training set (default: 1000 problems)

In [ ]:
from datasets import load_dataset

print("Downloading Codeforces-Python-Submissions from HuggingFace...")
print("(This may take a few minutes for the full 621K-row dataset)\n")

raw_dataset = load_dataset(HF_DATASET, split="train")
print(f"Loaded {len(raw_dataset):,} rows")
print(f"Columns: {raw_dataset.column_names}")

In [ ]:
import random

# ============== TUNABLE PARAMETERS ==============
MIN_RATING = 800
MAX_RATING = 1600
MAX_PROBLEMS = 1000  # Number of unique problems to keep
SEED = 42
# ================================================

random.seed(SEED)

# Filter for accepted solutions within the target difficulty range
filtered = raw_dataset.filter(
    lambda row: (
        row["verdict"] == "OK"
        and row["rating"] is not None
        and MIN_RATING <= row["rating"] <= MAX_RATING
        and row["test_cases"] is not None
        and len(row["test_cases"]) > 0
    ),
    num_proc=4,
)
print(f"After filtering (verdict=OK, rating {MIN_RATING}-{MAX_RATING}, has test_cases): {len(filtered):,} rows")

# Deduplicate: keep one accepted solution per unique problem (by contestId + index)
seen_problems = set()
deduped_indices = []
for i, row in enumerate(filtered):
    key = (row["contestId"], row["index"])
    if key not in seen_problems:
        seen_problems.add(key)
        deduped_indices.append(i)

deduped = filtered.select(deduped_indices)
print(f"After deduplication (one solution per problem): {len(deduped):,} unique problems")

# Sample down to MAX_PROBLEMS
if len(deduped) > MAX_PROBLEMS:
    sample_indices = sorted(random.sample(range(len(deduped)), MAX_PROBLEMS))
    final_dataset = deduped.select(sample_indices)
else:
    final_dataset = deduped

print(f"Final training set: {len(final_dataset):,} problems")

### Build Prompts & Export to JSONL

Each training row follows the OpenAI-compatible RFT format:

```json
{
  "messages": [
    {"role": "system", "content": "You are an expert competitive programmer..."},
    {"role": "user", "content": "<problem description>"}
  ],
  "metadata": {
    "test_cases": [{"input": "...", "output": "..."}, ...]
  }
}
```

The `metadata.test_cases` field is passed to the reward function so it can execute the model's code and verify correctness.

In [ ]:
SYSTEM_PROMPT = """You are an expert competitive programmer. Given a problem statement, write a correct and efficient Python solution.

Rules:
- Read input from stdin and write output to stdout
- Your solution must handle all edge cases
- Output your solution as a complete Python program inside a ```python code block"""

def build_problem_prompt(row):
    """Build a clean problem prompt from the dataset columns."""
    parts = []

    # Problem title and rating
    name = row.get("name", "Unknown")
    rating = row.get("rating", "?")
    parts.append(f"## {name} (Rating: {rating})\n")

    # Problem description
    desc = row.get("problem-description", "")
    if desc:
        parts.append(f"### Problem\n{desc}\n")

    # Input specification
    input_spec = row.get("input-specification", "")
    if input_spec:
        parts.append(f"### Input\n{input_spec}\n")

    # Output specification
    output_spec = row.get("output-specification", "")
    if output_spec:
        parts.append(f"### Output\n{output_spec}\n")

    # Example test cases from demo-input / demo-output
    demo_inputs = row.get("demo-input", [])
    demo_outputs = row.get("demo-output", [])
    if demo_inputs and demo_outputs:
        parts.append("### Examples\n")
        for i, (di, do) in enumerate(zip(demo_inputs, demo_outputs), 1):
            parts.append(f"**Example {i}:**")
            parts.append(f"Input:\n```\n{di.strip()}\n```")
            parts.append(f"Output:\n```\n{do.strip()}\n```\n")

    # Note
    note = row.get("note", "")
    if note:
        parts.append(f"### Note\n{note}\n")

    return "\n".join(parts)


# Preview a sample prompt
sample_row = final_dataset[0]
sample_prompt = build_problem_prompt(sample_row)
print("=== Sample Prompt (first 800 chars) ===")
print(sample_prompt[:800])
print("...")

In [ ]:
import os

LOCAL_DATA_DIR = "./tmp-data"
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

TRAINING_FILE_PATH = os.path.join(LOCAL_DATA_DIR, "codeforces_rft_train.jsonl")

num_written = 0
with open(TRAINING_FILE_PATH, "w") as f:
    for row in final_dataset:
        prompt = build_problem_prompt(row)

        # Build test cases for the reward function
        test_cases = row.get("test_cases", [])
        # Each test case should be {"input": "...", "output": "..."}
        formatted_test_cases = []
        for tc in test_cases:
            if isinstance(tc, dict) and "input" in tc and "output" in tc:
                formatted_test_cases.append({
                    "input": tc["input"].strip(),
                    "output": tc["output"].strip(),
                })

        record = {
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ],
            "metadata": {
                "test_cases": formatted_test_cases,
            },
        }

        f.write(json.dumps(record) + "\n")
        num_written += 1

print(f"Wrote {num_written:,} training samples to {TRAINING_FILE_PATH}")

# Show file size
size_mb = os.path.getsize(TRAINING_FILE_PATH) / (1024 * 1024)
print(f"File size: {size_mb:.1f} MB")

In [ ]:
# Preview a formatted training row
with open(TRAINING_FILE_PATH) as f:
    sample_line = json.loads(f.readline())

print("=== Formatted Training Row ===")
print(json.dumps(sample_line, indent=2)[:1000])
print("...")
print(f"\nNumber of test cases in metadata: {len(sample_line['metadata']['test_cases'])}")

---
## Step 4: Deploy the Lambda Reward Function

The reward function receives the model's generated code, extracts it, runs it against the test cases from `metadata`, and returns a score:

- **1.0** — all test cases pass
- **Partial credit** — fraction of test cases passed
- **0.1** — code was extracted but no tests passed (format reward)
- **0.0** — no code could be extracted from the response

In [ ]:
# Create Lambda execution role
lambda_client = session.client('lambda', region_name=AWS_REGION)
iam_client = session.client('iam', region_name=AWS_REGION)

print("Creating Lambda execution role...")

lambda_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "lambda.amazonaws.com"}, "Action": "sts:AssumeRole"}]
}

try:
    response = iam_client.create_role(
        RoleName=LAMBDA_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(lambda_trust_policy),
        Description=f"Execution role for {DATASET_NAME} reward function"
    )
    lambda_role_arn = response['Role']['Arn']
    iam_client.attach_role_policy(RoleName=LAMBDA_ROLE_NAME, PolicyArn='arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole')
    print(f"Created role: {LAMBDA_ROLE_NAME}")
    print("Waiting 10s for role propagation...")
    time.sleep(10)
except iam_client.exceptions.EntityAlreadyExistsException:
    lambda_role_arn = iam_client.get_role(RoleName=LAMBDA_ROLE_NAME)['Role']['Arn']
    print(f"Using existing role: {LAMBDA_ROLE_NAME}")

# Package and deploy Lambda
lambda_zip_content = create_lambda_deployment_package(
    source_file=REWARD_FUNCTION_FILE,
    zip_filename="lambda_deployment.zip",
    archive_name=f"{REWARD_FUNCTION_MODULE}.py"
)

print(f"\nDeploying Lambda: {LAMBDA_FUNCTION_NAME}...")
try:
    lambda_client.get_function(FunctionName=LAMBDA_FUNCTION_NAME)
    lambda_client.update_function_code(FunctionName=LAMBDA_FUNCTION_NAME, ZipFile=lambda_zip_content)
    waiter = lambda_client.get_waiter('function_updated_v2')
    waiter.wait(FunctionName=LAMBDA_FUNCTION_NAME)
    print("Updated existing function")
except lambda_client.exceptions.ResourceNotFoundException:
    lambda_client.create_function(
        FunctionName=LAMBDA_FUNCTION_NAME,
        Runtime='python3.11',
        Role=lambda_role_arn,
        Handler=f"{REWARD_FUNCTION_MODULE}.lambda_handler",
        Code={'ZipFile': lambda_zip_content},
        Timeout=300,
        MemorySize=512
    )
    print("Created new function")

waiter = lambda_client.get_waiter('function_active_v2')
waiter.wait(FunctionName=LAMBDA_FUNCTION_NAME)
lambda_arn = lambda_client.get_function(FunctionName=LAMBDA_FUNCTION_NAME)['Configuration']['FunctionArn']
print(f"Lambda ready: {lambda_arn}")

cleanup_lambda_deployment_package()

In [ ]:
# Create Bedrock role
print(f"Creating Bedrock role: {BEDROCK_ROLE_NAME}...")

bedrock_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "bedrock.amazonaws.com"}, "Action": "sts:AssumeRole"}]
}

bedrock_permissions = {
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow", "Action": "lambda:InvokeFunction", "Resource": lambda_arn}
    ]
}

try:
    response = iam_client.create_role(
        RoleName=BEDROCK_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(bedrock_trust_policy),
        Description="Execution role for Bedrock RFT"
    )
    bedrock_role_arn = response['Role']['Arn']
    print(f"Created role: {BEDROCK_ROLE_NAME}")
except iam_client.exceptions.EntityAlreadyExistsException:
    bedrock_role_arn = iam_client.get_role(RoleName=BEDROCK_ROLE_NAME)['Role']['Arn']
    print(f"Using existing role: {BEDROCK_ROLE_NAME}")

iam_client.put_role_policy(RoleName=BEDROCK_ROLE_NAME, PolicyName='BedrockRFTPermissions', PolicyDocument=json.dumps(bedrock_permissions))
print(f"Bedrock role ready: {bedrock_role_arn}")

### Test the Reward Function Locally

Before starting a training job, verify the reward function works correctly with a simulated trajectory.

In [ ]:
# Test the reward function locally by invoking the Lambda
test_payload = [
    {
        "id": "test_correct",
        "messages": [
            {"role": "user", "content": "Read two integers and print their sum."},
            {"role": "assistant", "content": "```python\na, b = map(int, input().split())\nprint(a + b)\n```"},
        ],
        "metadata": {
            "test_cases": [
                {"input": "2 3", "output": "5"},
                {"input": "0 0", "output": "0"},
            ]
        },
    },
    {
        "id": "test_wrong",
        "messages": [
            {"role": "user", "content": "Read two integers and print their sum."},
            {"role": "assistant", "content": "```python\nprint('hello world')\n```"},
        ],
        "metadata": {
            "test_cases": [
                {"input": "2 3", "output": "5"},
            ]
        },
    },
    {
        "id": "test_no_code",
        "messages": [
            {"role": "user", "content": "Read two integers and print their sum."},
            {"role": "assistant", "content": "I'm not sure how to solve this problem."},
        ],
        "metadata": {
            "test_cases": [
                {"input": "2 3", "output": "5"},
            ]
        },
    },
]

response = lambda_client.invoke(
    FunctionName=LAMBDA_FUNCTION_NAME,
    InvocationType='RequestResponse',
    Payload=json.dumps(test_payload),
)

results = json.loads(response['Payload'].read())
for r in results:
    score = r.get('aggregate_reward_score', 0)
    status = "PASS" if score > 0.5 else "FAIL"
    print(f"  [{status}] {r['id']}: score={score}")

print("\nReward function working correctly")

---
## Step 5: Create OpenAI Client with Bedrock API Key Authentication

In [ ]:
from aws_bedrock_token_generator import provide_token

TARGET_ROLE_ARN = BEDROCK_ROLE_NAME
TARGET_ACCOUNT_ID = boto3.client('sts').get_caller_identity().get('Account')

MANTLE_ENDPOINT = f"https://bedrock-mantle.{AWS_REGION}.api.aws"

ST_BEDROCK_API_KEY = provide_token(region=AWS_REGION)

# Fine-tuning configuration
MODEL_ID = "nvidia.nemotron-nano-3-30b"

print(f"Target Role: {TARGET_ROLE_ARN}")
print(f"Account ID: {TARGET_ACCOUNT_ID}")
print(f"Region: {AWS_REGION}")
print(f"Endpoint: {MANTLE_ENDPOINT}")
print(f"Model: {MODEL_ID}")
print(f"Training File: {TRAINING_FILE_PATH}")

In [ ]:
from openai import OpenAI
import urllib3

def create_openai_client(endpoint, region, account_id=None, verify_ssl=False):
    """Create OpenAI client with Bedrock API key authentication."""
    return OpenAI(
        base_url=f"{endpoint}/v1",
        api_key=ST_BEDROCK_API_KEY,
    )

# Suppress SSL warnings if not verifying SSL
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

client = create_openai_client(
    endpoint=MANTLE_ENDPOINT,
    region=AWS_REGION,
    account_id=TARGET_ACCOUNT_ID,
    verify_ssl=False  # Set to True for production
)

print(f"OpenAI client created successfully")
print(f"  Base URL: {MANTLE_ENDPOINT}/v1")
print(f"  Model: {MODEL_ID}")

---
## Step 6: Fine-Tuning API Operations

### API Operation 1: List Fine-Tuning Jobs

In [ ]:
print("=" * 80)
print("API Operation 1: List Fine-Tuning Jobs")
print("=" * 80)
print()

response = client.fine_tuning.jobs.list(limit=10)
print(json.dumps(response.model_dump(), indent=2))

### API Operation 2: Upload Training File

In [ ]:
print("=" * 80)
print("API Operation 2: Upload Training File")
print("=" * 80)
print()
print(f"Uploading: {TRAINING_FILE_PATH}")
print()

with open(TRAINING_FILE_PATH, 'rb') as f:
    file_response = client.files.create(
        file=f,
        purpose='fine-tune'
    )

print(json.dumps(file_response.model_dump(), indent=2))

training_file_id = file_response.id
print(f"\nTraining file uploaded: {training_file_id}")

### List & Retrieve Files

In [ ]:
# List files
files_response = client.files.list(purpose='fine-tune')
print(json.dumps(files_response.model_dump(), indent=2))

In [ ]:
# Retrieve file details
file_details = client.files.retrieve(training_file_id)
print(json.dumps(file_details.model_dump(), indent=2))

### API Operation 3: Create Fine-Tuning Job (RFT)

**Hyperparameters:**
- **n_epochs** — Number of full passes through the training data. Start with 1; increase if rewards are still improving.
- **batch_size** — Prompts per training step. Larger = more stable updates, smaller = faster iteration.
- **learning_rate_multiplier** — Scales the default learning rate. >1.0 for faster learning, <1.0 for more stability.

In [ ]:
print("=" * 80)
print("API Operation 3: Create Fine-Tuning Job (RFT)")
print("=" * 80)
print()

job_response = client.fine_tuning.jobs.create(
    model=MODEL_ID,
    training_file=training_file_id,
    extra_body={
        "method": {
            "type": "reinforcement",
            "reinforcement": {
                "grader": {
                    "type": "lambda",
                    "lambda": {
                        "function": lambda_arn
                    }
                },
                "hyperparameters": {
                    "n_epochs": 1,
                    "batch_size": 4,
                    "learning_rate_multiplier": 1.0
                }
            }
        }
    }
)

print(json.dumps(job_response.model_dump(), indent=2))

job_id = job_response.id
print(f"\nFine-tuning job created: {job_id}")

### API Operation 4: List Jobs with Pagination

In [ ]:
response = client.fine_tuning.jobs.list(limit=20)
print(json.dumps(response.model_dump(), indent=2))

In [ ]:
from datetime import datetime, timezone

response = client.fine_tuning.jobs.list(limit=200)

jobs = []
for job in response.data:
    details = client.fine_tuning.jobs.retrieve(job.id)
    d = details.model_dump()

    created = d["created_at"]
    finished = d["finished_at"]
    duration = None
    if created and finished:
        duration = finished - created

    jobs.append({
        "Job ID": d["id"],
        "Model": d["model"],
        "Created (UTC)": datetime.fromtimestamp(created, tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
        "Duration (min)": round(duration / 60, 1) if duration else "N/A",
        "Status": d["status"],
    })

if jobs:
    header = jobs[0].keys()
    col_widths = {k: max(len(k), max(len(str(j[k])) for j in jobs)) for k in header}
    fmt = " | ".join(f"{{:<{col_widths[k]}}}" for k in header)
    sep = "-+-".join("-" * col_widths[k] for k in header)

    print(fmt.format(*header))
    print(sep)
    for j in jobs:
        print(fmt.format(*[str(j[k]) for k in header]))
else:
    print("No jobs found.")

### API Operation 5: Describe Specific Job

In [ ]:
print(f"Getting details for job: {job_id}")
print()

job_details = client.fine_tuning.jobs.retrieve(job_id)
print(json.dumps(job_details.model_dump(), indent=2))
print(f"\nRFT job is currently: {job_details.status}")

### API Operation 6: List Events

In [ ]:
print(f"Listing events for job: {job_id}")
print()

events_response = client.fine_tuning.jobs.list_events(
    fine_tuning_job_id=job_id,
    limit=100
)

print(json.dumps(events_response.model_dump(), indent=2))

### Plot Training Metrics

In [ ]:
import matplotlib.pyplot as plt

metrics = [e.data for e in events_response.data if e.type == "metrics"]

if metrics:
    steps = [m["step"] for m in metrics]
    fields = ["critic_rewards_mean", "actor_pg_loss", "actor_entropy",
              "actor_grad_norm", "critic_advantages_mean", "response_length_mean"]

    fig, axes = plt.subplots(3, 2, figsize=(14, 10))
    fig.suptitle("Nemotron Codeforces RFT — Training Metrics", fontsize=14)

    for ax, field in zip(axes.flat, fields):
        values = [m[field] for m in metrics]
        ax.plot(steps, values, linewidth=1.2)
        ax.set_title(field)
        ax.set_xlabel("Step")
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("No metrics available yet — job may still be starting.")

### How to Read These Metrics

| Metric | Meaning |
|---|---|
| **critic_rewards_mean** | Average reward score across the batch. For code problems, this reflects the fraction of test cases the model's code passes. Watch for this trending upward. |
| **actor_pg_loss** | Policy gradient loss — the objective being optimized. Fluctuates naturally. |
| **actor_entropy** | How diverse the model's outputs are. If it collapses toward 0, the model is becoming too deterministic (mode collapse). Should decrease gradually. |
| **actor_grad_norm** | Magnitude of gradient updates. Large spikes indicate instability. |
| **critic_advantages_mean** | How much better/worse responses are vs. the critic's baseline. Near-zero means well-calibrated. |
| **response_length_mean** | Average token length of generated responses. Monitor for unbounded growth (reward hacking). |

**What to watch for:**
- `critic_rewards_mean` trending upward = model is learning to write better code
- `actor_entropy` collapsing to 0 = mode collapse (bad)
- `response_length_mean` exploding = the model may be gaming length for reward

### API Operation 7: List Checkpoints

In [ ]:
print(f"Listing checkpoints for job: {job_id}")
print()

try:
    checkpoints_response = client.fine_tuning.jobs.checkpoints.list(
        fine_tuning_job_id=job_id
    )
    print(json.dumps(checkpoints_response.model_dump(), indent=2))

    if checkpoints_response.data:
        print(f"\nFound {len(checkpoints_response.data)} checkpoint(s)")
    else:
        print("No checkpoints available yet (job may still be running)")
except Exception as e:
    print(f"Error listing checkpoints: {e}")
    print("Checkpoints are only available after the job starts training")

---
## Additional API Operations

### Cancel a Fine-Tuning Job

In [ ]:
import warnings

# Uncomment to cancel a job
# cancel_response = client.fine_tuning.jobs.cancel(job_id)
# print(json.dumps(cancel_response.model_dump(), indent=2))

warnings.warn("To cancel a job, uncomment the code above and replace job_id")

---
## Step 7: Run Inference with Fine-Tuned Model

Once the job completes, test the fine-tuned model on a competitive programming problem.

In [ ]:
job_details = client.fine_tuning.jobs.retrieve(job_id)
job_details

In [ ]:
%%time
if job_details.status == 'succeeded' and job_details.fine_tuned_model:
    fine_tuned_model = job_details.fine_tuned_model
    print(f"Using fine-tuned model: {fine_tuned_model}")
    print()

    test_problem = """## Two Sum (Rating: 800)

### Problem
You are given two integers a and b. Print their sum.

### Input
The first line contains two integers a and b (1 <= a, b <= 10^9).

### Output
Print a single integer — the sum of a and b.

### Examples
**Example 1:**
Input:
```
2 3
```
Output:
```
5
```"""

    inference_response = client.chat.completions.create(
        model=fine_tuned_model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": test_problem},
        ],
        max_tokens=512
    )

    print(json.dumps(inference_response.model_dump(), indent=2))
else:
    print(f"Job status: {job_details.status}")
    print("Job must be in 'succeeded' status to run inference")

### Test Streaming Response

In [ ]:
from colorama import Fore, Style

codeforces_problem = """## Watermelon (Rating: 800)

### Problem
One hot summer day Pete and his friend Billy found a watermelon. They decided to split it in two parts, but each part must have an even weight. Determine if it is possible to do so.

### Input
The first line contains a single integer w (1 <= w <= 100) — the weight of the watermelon.

### Output
Print "YES" if it is possible to split the watermelon into two parts, each with even weight, and "NO" otherwise.

### Examples
**Example 1:**
Input:
```
8
```
Output:
```
YES
```"""

stream = client.responses.create(
    model=fine_tuned_model,
    input=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": codeforces_problem},
    ],
    stream=True,
    reasoning={"effort": "low"}
)

for event in stream:
    if event.type in ['response.reasoning_part.added', 'response.reasoning_part.done']:
        print(Fore.GREEN + Style.DIM + "\n<thinking>\n")
    if event.type == 'response.reasoning_text.delta':
        print(Fore.GREEN + Style.DIM + event.delta, end="", flush=True)
    if event.type in ['response.output_text.delta']:
        print(Fore.BLACK + Style.RESET_ALL + event.delta, end="", flush=True)

---
## (Optional) Benchmarking Base vs. Fine-Tuned Model Performance

Compare the base Nemotron Nano 3 30B model against the fine-tuned version across three dimensions: time to first token (TTFT), output throughput (tokens per second), and total latency.

In [ ]:
import time
import tiktoken
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

enc = tiktoken.get_encoding("cl100k_base")

def benchmark_model(client, model_id, prompt, label="model"):
    stream = client.responses.create(
        model=model_id,
        input=[{"role": "user", "content": prompt}],
        stream=True,
        reasoning={"effort": "low"}
    )

    start = time.perf_counter()
    ttft = None
    first_output_token_time = None
    reasoning_text = ""
    output_text = ""

    for event in stream:
        now = time.perf_counter()
        if event.type == 'response.reasoning_text.delta':
            if ttft is None:
                ttft = now - start
            reasoning_text += event.delta
        elif event.type == 'response.output_text.delta':
            if first_output_token_time is None:
                first_output_token_time = now - start
            output_text += event.delta

    end = time.perf_counter()
    total_time = end - start

    reasoning_tokens = len(enc.encode(reasoning_text))
    output_tokens = len(enc.encode(output_text))
    total_tokens = reasoning_tokens + output_tokens

    gen_duration = total_time - (ttft if ttft else 0)
    output_duration = total_time - (first_output_token_time if first_output_token_time else 0)

    result = {
        "label": label,
        "ttft": ttft or 0,
        "time_to_first_output": first_output_token_time or 0,
        "total_time": total_time,
        "reasoning_tokens": reasoning_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
        "total_tokens_per_sec": total_tokens / gen_duration if gen_duration > 0 else 0,
        "output_tokens_per_sec": output_tokens / output_duration if output_duration > 0 else 0,
    }

    print(f"\n{'=' * 60}")
    print(f"  {label}")
    print(f"{'=' * 60}")
    print(f"  TTFT (first reasoning token):  {result['ttft']:.3f}s")
    print(f"  Time to first output token:    {result['time_to_first_output']:.3f}s")
    print(f"  Total time:                    {result['total_time']:.3f}s")
    print(f"  Reasoning tokens:              {result['reasoning_tokens']}")
    print(f"  Output tokens:                 {result['output_tokens']}")
    print(f"  Total tokens:                  {result['total_tokens']}")
    print(f"  Total tokens/sec:              {result['total_tokens_per_sec']:.1f}")
    print(f"  Output tokens/sec:             {result['output_tokens_per_sec']:.1f}")
    print(f"{'=' * 60}\n")

    return result

# Run benchmarks
bench_prompt = "Write a Python solution to find the longest increasing subsequence in an array. Include time complexity analysis."
base_result = benchmark_model(client, MODEL_ID, bench_prompt, label="Base Model")
ft_result = benchmark_model(client, fine_tuned_model, bench_prompt, label="Fine-Tuned Model")

In [ ]:
# Plot benchmark comparison
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle("Base vs Fine-Tuned Nemotron — Inference Performance",
             fontsize=15, fontweight='bold', y=1.02)

colors = ["#2563EB", "#F97316"]
labels = ["Base", "Fine-Tuned"]

def pct_change(base, ft):
    if base == 0:
        return 0
    return ((ft - base) / base) * 100

def annotate_change(ax, base_val, ft_val, y_offset_frac=0.15, higher_is_better=False):
    change = pct_change(base_val, ft_val)
    is_improvement = (change > 0 and higher_is_better) or (change < 0 and not higher_is_better)
    color = "#16A34A" if is_improvement else "#DC2626"
    arrow_symbol = "\u25b2" if change > 0 else "\u25bc"
    sign = "+" if change > 0 else ""

    max_val = max(base_val, ft_val)
    y_pos = max_val + max_val * y_offset_frac

    ax.annotate(
        f"{arrow_symbol} {sign}{change:.1f}%",
        xy=(0.5, y_pos), fontsize=12, fontweight='bold',
        color=color, ha='center', va='bottom',
        bbox=dict(boxstyle="round,pad=0.3", facecolor=color, alpha=0.12,
                  edgecolor=color, linewidth=1.2)
    )

# Panel 1: TTFT
vals = [base_result["ttft"], ft_result["ttft"]]
bars = axes[0].bar(labels, vals, color=colors, width=0.5, edgecolor='white', linewidth=1.5)
axes[0].set_title("Time to First Token", fontweight='bold', pad=12)
axes[0].set_ylabel("Seconds (lower is better)", fontsize=10, color="#666")
for bar, v in zip(bars, vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + max(vals)*0.01,
                 f"{v:.3f}s", ha='center', va='bottom', fontweight='bold', fontsize=11)
annotate_change(axes[0], vals[0], vals[1], higher_is_better=False)

# Panel 2: Output Tokens/sec
vals = [base_result["output_tokens_per_sec"], ft_result["output_tokens_per_sec"]]
bars = axes[1].bar(labels, vals, color=colors, width=0.5, edgecolor='white', linewidth=1.5)
axes[1].set_title("Output Throughput", fontweight='bold', pad=12)
axes[1].set_ylabel("Tokens/sec (higher is better)", fontsize=10, color="#666")
for bar, v in zip(bars, vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, v + max(vals)*0.01,
                 f"{v:.1f}", ha='center', va='bottom', fontweight='bold', fontsize=11)
annotate_change(axes[1], vals[0], vals[1], higher_is_better=True)

# Panel 3: Total Time
vals = [base_result["total_time"], ft_result["total_time"]]
bars = axes[2].bar(labels, vals, color=colors, width=0.5, edgecolor='white', linewidth=1.5)
axes[2].set_title("Total Latency", fontweight='bold', pad=12)
axes[2].set_ylabel("Seconds (lower is better)", fontsize=10, color="#666")
for bar, v in zip(bars, vals):
    axes[2].text(bar.get_x() + bar.get_width()/2, v + max(vals)*0.01,
                 f"{v:.2f}s", ha='center', va='bottom', fontweight='bold', fontsize=11)
annotate_change(axes[2], vals[0], vals[1], higher_is_better=False)

for ax in axes:
    ymin, ymax = ax.get_ylim()
    ax.set_ylim(0, ymax * 1.4)
    ax.tick_params(axis='x', labelsize=11)

legend_elements = [mpatches.Patch(facecolor=colors[0], label='Base Model'),
                   mpatches.Patch(facecolor=colors[1], label='Fine-Tuned Model')]
fig.legend(handles=legend_elements, loc='upper left', frameon=True,
           fontsize=10, edgecolor='#ccc', bbox_to_anchor=(0.98, 0.98))

plt.tight_layout()
plt.show()

---
## Cleanup

In [ ]:
import shutil

if os.path.exists(LOCAL_DATA_DIR):
    shutil.rmtree(LOCAL_DATA_DIR)
    print(f"Cleaned up {LOCAL_DATA_DIR}")

---
## Summary

This notebook demonstrated end-to-end reinforcement fine-tuning (RFT) of Nemotron Nano 3 30B on Codeforces competitive programming problems via Bedrock's OpenAI-compatible APIs. We:

1. Downloaded and preprocessed the Codeforces-Python-Submissions dataset from HuggingFace, filtering for accepted solutions on beginner-to-intermediate problems (rating 800–1600)
2. Formatted the data as OpenAI-compatible JSONL with problem prompts and test cases in metadata
3. Deployed a Lambda reward function that extracts Python code from model responses and runs it against test cases for automated scoring
4. Uploaded training data and kicked off an RFT job using the OpenAI SDK
5. Monitored training metrics and ran inference against the fine-tuned model

For more information, visit the documentation: https://docs.aws.amazon.com/bedrock/latest/userguide/fine-tuning-openai-apis.html